# 🛒 Physical Retail Layout & Spatial Infrastructure Analytics
> **Sub-title:** Optimizing Store Architecture, Entrance Sensory Priming, and Checkout Real Estate Using Transaction Sequence Data

---

## 🎯 1. Executive Summary & Problem Framing 

### 1.1 The Business Problem
Brick-and-mortar retail managers face a major analytical barrier: traditional point-of-sale (POS) and transaction logs record *what* products were purchased, but lack physical floor plan blueprints, indoor GPS tracking, overhead camera feeds, or Bluetooth beacons. As a result, retailers struggle to:
* Measure how customers physically navigate store aisles.
* Quantify the revenue impact of entrance sensory displays.
* Evaluate front-end register queue performance without committing millions of dollars to expensive hardware infrastructure.

### 1.2 The Spatial Proxy Solution
To solve this physical tracking gap at **zero hardware cost**, this project engineered a **Spatial Data Proxy** leveraging transaction pick sequence (`add_to_cart_order`). Because shoppers move sequentially through physical store space over time, cart pick order acts as a direct proxy for customer navigation flow from store entry to checkout register queues.
Using **Python** (data cleaning), **DuckDB** (spatial SQL feature engineering), and **Tableau** (interactive dashboards), we algorithmically reconstructed three functional store zones:
* **Entrance Sensory Zone (`add_to_cart_order ≤ 3`):** High-sensory, fresh departments (*Produce, Bakery, Deli, Floral\*).
* **Main Aisle Staple Zone (`3 < add_to_cart_order < 8`):** Core center-aisle grocery categories (\*Dry Goods, Dairy, Frozen*).
* **Checkout Impulse Zone (`add_to_cart_order ≥ 8`):** High-margin grab-and-go products (*Snacks, Beverages*) adjacent to register queues.

---

### 💡 Key Analytical Discoveries
1. **Entrance Priming Effect:** Shoppers who select fresh items first build **21.0% larger total baskets** (10.73 vs. 8.87 items/cart). 
2. **Navigation Flow Sequence:** Customers follow a predictable path, executing planned destination picks early (*Alcohol at avg pick #5.38*) before navigating center-aisle staples.
3. **Category Anchors:** *Produce* and *Dairy/Eggs* account for **\~46% of total store item volume**, serving as primary perimeter traffic magnets.
4. **Checkout Queue Impulse:** Front-end register displays capture **6.98% of total store item volume** at a late-trip pick position of **14.39**, converting captive queue waiting time into high-margin sales.

## Step 1: Data Ingestion & Schema Inspection 

I loaded three foundational CSV datasets from the Instacart Market Basket dataset to inspect their structures and key attributes:

- `order_products__prior.csv`: Captures shopping trip sequence details (`order_id`, `product_id`, `add_to_cart_order`).

- `products.csv`: Links specific product names to their department IDs.

- `departments.csv`: Defines store departments (e.g., *bakery*, *produce*, *snacks*, *frozen*).

In [ ]:
import pandas as pd

# 1. Loading the 3 CSV files
orders = pd.read_csv("Data/order_products__prior.csv")
products = pd.read_csv("Data/products.csv")
departments = pd.read_csv("Data/departments.csv")

# 2. Checking the first few rows of each table
print("=== ORDER PRODUCTS (Sequences) ===")
print(orders[["order_id", "product_id", "add_to_cart_order"]].head())

print('\n=== PRODUCTS ===')
print(products[["product_id", "product_name", "department_id"]].head())

print('\n=== DEPARTMENTS ===')
print(departments.head())

=== ORDER PRODUCTS (Sequences) ===
   order_id  product_id  add_to_cart_order
0         2       33120                  1
1         2       28985                  2
2         2        9327                  3
3         2       45918                  4
4         2       30035                  5

=== PRODUCTS ===
   product_id                                       product_name  \
0           1                         Chocolate Sandwich Cookies   
1           2                                   All-Seasons Salt   
2           3               Robust Golden Unsweetened Oolong Tea   
3           4  Smart Ones Classic Favorites Mini Rigatoni Wit...   
4           5                          Green Chile Anytime Sauce   

   department_id  
0             19  
1             13  
2              7  
3              1  
4             13  

=== DEPARTMENTS ===
   department_id department
0              1     frozen
1              2      other
2              3     bakery
3              4    produce
4    

### 💡 Key Takeaway — Data Verification

- **Data Integrity:** I successfully loaded and inspected the three primary relational tables (`orders`, `products`, and `departments`).

- **Sequential Tracking:** The `add_to_cart_order` field provides an explicit numerical sequence of when each item entered a customer's cart during their store trip, serving as our proxy for physical customer navigation.

### 🛍️ Retail Store Layout Design & Zone Definitions 

In physical grocery retail, store layouts are strategically designed around human psychology and shopper movement patterns. To translate digital transaction sequences into physical store movement, we categorize cart order (`add_to_cart_order`) into three core physical zones:

1. **`Entrance_Sensory_Zone`**
    - **Criteria:** Fresh, sensory departments (*Bakery, Produce, Deli, Floral*) added early in the trip (`add_to_cart_order <= 3`).
    - **Retail Psychology:** Stores place visual and aromatic fresh departments right at the front entrance to lower customer stress, boost mood, and establish a high-quality perception of the store immediately upon arrival.

2. **`Main_Store_Aisles`**
    - **Criteria:** Core middle-aisle categories (*Pantry, Frozen, Canned Goods, Household*) added mid-trip (`3 < add_to_cart_order < 8`).
    - **Retail Psychology:** This represents the main body of the store where shoppers spend the majority of their trip browsing standard grocery aisles and filling baseline household needs.

3. **`Checkout_Impulse_Zone`**
    - **Criteria:** Grab-and-go categories (*Snacks, Beverages*) added late in the trip (`add_to_cart_order >= 8`).
    - **Retail Psychology:** Placed near register queues to capitalize on decision fatigue at the end of a visit, encouraging high-margin, last-minute additions right before payment.

## Step 2: Spatial Feature Engineering & Zone Mapping

I merged my transactional tables together and engineered a new spatial feature, `store_zone`, based on item cart sequence (`add_to_cart_order`) and department type:

- **`Entrance_Sensory_Zone`**: High-sensory fresh departments (*bakery*, *produce*, *deli*, *floral*) selected early in the trip (items 1–3). 

- **`Checkout_Impulse_Zone`**: Impulse categories (*snacks*, *beverages*) selected late in the trip (items 8+). 

- **`Main_Store_Aisles`**: Core mid-trip store navigation across all remaining categories.  (items 4 to 7)

Finally, I exported a processed sample to `Data/spatial_transactions_sample.csv` for local SQL analytics.

In [7]:
import pandas as pd

# 1. Loading the CSV files
orders = pd.read_csv("Data/order_products__prior.csv")
products = pd.read_csv("Data/products.csv")
departments = pd.read_csv("Data/departments.csv")

# 2. Merging the tables together
df = orders.merge(products, on="product_id", how="left")
df = df.merge(departments, on="department_id", how="left")

# 3. Defining my Physical Store Zone mapping function
def assign_store_zone(row):
    dept = str(row['department']).lower()
    seq = row["add_to_cart_order"]

    if dept in ["bakery", "produce", "deli", "floral"] and seq <= 3:
        return "Entrance_Sensory_Zone"
    elif dept in ["snacks", "beverages"] and seq >= 8:
        return "Checkout_Impulse_Zone"
    else:
        return "Main_Store_Aisles"

# 4. Applying the mapping to create a new 'store_zone' column
df["store_zone"] = df.apply(assign_store_zone, axis=1)

# 5. Inspecting a sample of my mapped data
print("=== MAPPED SPATIAL TRANSACTIONS ===")
print(df[["order_id", "product_name", "department", "add_to_cart_order", "store_zone"]].head(10))

# 6. Saving a processed sample CSV for my BigQuery step
df.head(50000).to_csv("Data/spatial_transactions_sample.csv", index=False)
print("\nSuccess! Saved 'Data/spatial_transactions_sample.csv'")

=== MAPPED SPATIAL TRANSACTIONS ===
   order_id                                       product_name  department  \
0         2                                 Organic Egg Whites  dairy eggs   
1         2                              Michigan Organic Kale     produce   
2         2                                      Garlic Powder      pantry   
3         2                                     Coconut Butter      pantry   
4         2                                  Natural Sweetener      pantry   
5         2                                            Carrots     produce   
6         2                   Original Unflavored Gelatine Mix      pantry   
7         2           All Natural No Stir Creamy Almond Butter      pantry   
8         2                            Classic Blend Cole Slaw     produce   
9         3  Total 2% with Strawberry Lowfat Greek Strained...  dairy eggs   

   add_to_cart_order             store_zone  
0                  1      Main_Store_Aisles  
1            

### 💡 Key Takeaway — Spatial Zone Creation

- **Spatial Mapping:** We transformed raw transactional sequence records into three distinct physical retail zones: `Entrance_Sensory_Zone`, `Main_Store_Aisles`, and `Checkout_Impulse_Zone`.

- **Sample Export:** We successfully compiled and exported a 50,000-row sample (`spatial_transactions_sample.csv`) for local high-speed SQL analytics in DuckDB.

### 🔬 Core Methodology: Spatial Data Proxy & Behavioral Assumptions 

#### 1. The Cart Order Data Proxy (`add_to_cart_order`)
- **Physical Limitation:** Standard transactional logs lack physical store floor plan blueprints, GPS coordinates, or video camera movement feeds. 
- **Analytical Solution:** We establish `add_to_cart_order` (the numerical sequence in which items enter a cart) as a **data proxy for physical navigation through space and time**.
- **Methodological Justification:** By combining cart pick sequence numbers with industry-standard retail layout conventions (e.g., fresh produce near entrances; single-serve snacks near registers), sequence order reliably reflects a shopper's progression from store entry to cash register checkout.

#### 2. Interpreting Sequence Numbers & Behavioral Intent 
- **Early Pick Sequence (`add_to_cart_order < 6`):** Indicates **High Destination Intent**. For example, an average pick order of **`5.38` for Alcohol** demonstrates that shoppers navigate directly to that department early in their visit with planned intent, rather than making an accidental or late-trip impulse pick.
- **Mid-Trip Sequence (`6 <= add_to_cart_order < 12`):** Represents **Exploratory Center-Aisle Navigation**. Shoppers spend the bulk of their trip browsing staple grocery categories across main aisles.
- **Late-Trip Sequence (`add_to_cart_order >= 12`):** Represents **Front-End Queue &amp; Impulse Behavior**. Items added near position **`14.39`** reflect last-minute decisions made while waiting in register queues.

## Step 3: Baseline SQL Analysis — Physical Zone Summary 

I used DuckDB to execute a local SQL query against `Data/spatial_transactions_sample.csv`. This aggregates total item counts and calculates the average item pick sequence (`avg_item_pick_order`) for each physical store zone to verify that my spatial mapping accurately distinguishes early entrance picks from late impulse buys.

In [11]:
import duckdb

# Running SQL directly on my CSV file with clear column names
query = """
SELECT
    store_zone AS physical_store_zone,
    COUNT(order_id) AS total_items_purchased,
    ROUND(AVG(add_to_cart_order), 2) AS avg_item_pick_order
FROM 'Data/spatial_transactions_sample.csv'
GROUP BY store_zone;
"""

duckdb.sql(query).show()

┌───────────────────────┬───────────────────────┬─────────────────────┐
│  physical_store_zone  │ total_items_purchased │ avg_item_pick_order │
│        varchar        │         int64         │       double        │
├───────────────────────┼───────────────────────┼─────────────────────┤
│ Main_Store_Aisles     │                 41402 │                8.39 │
│ Entrance_Sensory_Zone │                  5106 │                 2.0 │
│ Checkout_Impulse_Zone │                  3492 │               14.39 │
└───────────────────────┴───────────────────────┴─────────────────────┘



### 💡 Key Takeaway — Physical Zone Behaviors

- **Entrance Validation (`avg_item_pick_order = 2.00`):** Fresh, sensory items (bakery, produce, deli) are picked up almost immediately at the start of the trip (averaging item #2). 
- **Mid-Trip Bulk (`avg_item_pick_order = 8.39`):** Over 82% of total cart volume (41,402 items) is gathered mid-trip across the main store aisles.

- **Checkout Timing (`avg_item_pick_order = 14.39`):** Impulse purchases occur nearly 7x later in sequence position than entrance picks, confirming a distinct end-of-trip purchasing phase before payment.

## Step 4: Question 1 — Testing the Entrance Priming Effect

I tested whether visiting high-sensory fresh departments (bakery, produce, deli) at the start of a shopping trip correlates with larger overall basket sizes.

We use a SQL Common Table Expression (CTE) to categorize each order into two shopper groups:

1. **Visited Entrance Zone First:** Orders containing an entrance sensory item. 
2. **Skipped Entrance Zone:** Orders that bypassed the entrance and went directly to main aisles.

In [ ]:
import duckdb
# Query 2: Entrance Priming Effect Analysis
query_entrance = """
WITH OrderSummary AS (
     SELECT
        order_id,
        COUNT(*) AS total_basket_items,
        MAX(CASE WHEN store_zone = 'Entrance_Sensory_Zone' THEN 1 ELSE 0 END) AS visited_entrance_first
        FROM 'Data/spatial_transactions_sample.csv'
        GROUP BY order_id
)

SELECT
    CASE
        WHEN visited_entrance_first = 1 THEN 'Visited Entrance Zone First'
        ELSE 'Skipped Entrance Zone'
        END AS shopper_group,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(total_basket_items), 2) AS avg_basket_size
    FROM OrderSummary
    GROUP BY visited_entrance_first
    ORDER BY avg_basket_size DESC;
"""

duckdb.sql(query_entrance).show()

┌─────────────────────────────┬──────────────┬─────────────────┐
│        shopper_group        │ total_orders │ avg_basket_size │
│           varchar           │    int64     │     double      │
├─────────────────────────────┼──────────────┼─────────────────┤
│ Visited Entrance Zone First │         3142 │           10.73 │
│ Skipped Entrance Zone       │         1836 │            8.87 │
└─────────────────────────────┴──────────────┴─────────────────┘



### 💡 Key Takeaway — The Entrance Priming Effect
- **Basket Size Expansion (10.73 vs. 8.87 items):** Shoppers who start their trip in the `Entrance_Sensory_Zone` build baskets that are **\~21% larger** (+1.86 items per cart) than those who bypass fresh entrance departments.
- **Order Volume Distribution:** Over **63% of orders** (3,142 out of 4,978) engage with fresh sensory categories at the beginning of their trip, demonstrating that entrance displays serve as a primary starting anchor for the majority of store visits.
- **Strategic Impact:** Prioritising high-quality fresh produce, bakery, and floral displays at the store front actively primes shoppers to spend more across the rest of the store floor.

## Step 5: Question 4 — Department Popularity (Anchors vs. Low-Volume Zones)

I aggregated total item purchases by department to identify high-volume "anchor departments" (that draw primary traffic) versus lower-volume departments. 

I also included average pick sequence (`avg_item_pick_order`) to understand when during the trip each category is selected.

In [ ]:
# Query 3: Department Volume & Pick Order Ranking
query_departments = """
SELECT
    department AS department_name,
    COUNT(*) AS total_items_purchased,
    ROUND(AVG(add_to_cart_order), 2) AS avg_item_pick_order
FROM 'Data/spatial_transactions_sample.csv'
GROUP BY department
ORDER BY total_items_purchased DESC;
"""

duckdb.sql(query_departments).show()

┌─────────────────┬───────────────────────┬─────────────────────┐
│ department_name │ total_items_purchased │ avg_item_pick_order │
│     varchar     │         int64         │       double        │
├─────────────────┼───────────────────────┼─────────────────────┤
│ produce         │                 14542 │                7.86 │
│ dairy eggs      │                  8443 │                7.47 │
│ snacks          │                  4456 │                8.94 │
│ beverages       │                  4139 │                 6.9 │
│ frozen          │                  3398 │                8.47 │
│ pantry          │                  2899 │                9.16 │
│ bakery          │                  1878 │                7.85 │
│ canned goods    │                  1669 │                9.43 │
│ deli            │                  1572 │                 8.7 │
│ dry goods pasta │                  1286 │                9.87 │
│ household       │                  1124 │                7.89 │
│ breakfas

### 💡 Key Takeaway — Department Popularity & Pick Order Sequence
- **Primary Traffic Anchors:** **Produce** (14,542 items) and **Dairy & Eggs** (8,443 items) represent nearly **46% of all item purchases**, acting as major destination anchors that drive foot traffic across the store floor. 

- **Trip Sequence & Timing Insights:** 
    - **Alcohol (`5.38`)** is picked earliest, indicating planned, deliberate choices right after entry. 
    - **Beverages (`6.90`)** are selected significantly earlier than **Snacks (`8.94`)**, showing that shoppers navigate liquid refreshment aisles mid-trip before picking up dry snacks later. 
    - **Dry Goods & Pasta (`9.87`)** and **Babies (`11.00`)** are added near the end of main aisle navigation, serving as secondary destination picks.

## Step 6: Question 2 — Analyzing Checkout Impulse Contribution 

We measure the proportion of total store volume originating from late-trip impulse purchases in the `Checkout_Impulse_Zone`. We evaluate how significantly end-of-trip snack and beverage picks contribute to overall basket volume.

In [16]:
import duckdb
# Query 4: Checkout Impulse Volume & Percentage Share
query_impulse = """
SELECT
    store_zone,
    COUNT(*) AS total_items_purchased,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM 'Data/spatial_transactions_sample.csv'), 2) AS pct_total_volume
    FROM 'Data/spatial_transactions_sample.csv'
    GROUP BY store_zone
    ORDER BY total_items_purchased DESC;
    """ 

duckdb.sql(query_impulse).show()

┌───────────────────────┬───────────────────────┬──────────────────┐
│      store_zone       │ total_items_purchased │ pct_total_volume │
│        varchar        │         int64         │      double      │
├───────────────────────┼───────────────────────┼──────────────────┤
│ Main_Store_Aisles     │                 41402 │             82.8 │
│ Entrance_Sensory_Zone │                  5106 │            10.21 │
│ Checkout_Impulse_Zone │                  3492 │             6.98 │
└───────────────────────┴───────────────────────┴──────────────────┘



## 💡 Key Takeaway — Front-End Queue & Checkout Impulse Contribution 
- **Volume Impact (6.98% of Total Volume):** Late-trip impulse categories (*Snacks* and *Beverages* picked at `add_to_cart_>= 8`) account for **3,492 items** (6.98% of total store purchases). 
- **End-of-Trip Timing (14.39 Average Pick Order):** Impulse purchases occur at an average pick order of **14.39**, confirming that customers select these items while waiting in checkout register queues. 
- **Architectural & Merchandising Value:** Dedicating front-of-store real estate to high-margin impulse displays successfully captures last-minute basket expansion driven by end-of-trip decision fatigue.

## 🛠️ Step 7: Data Export & Preparation for Tableau Dashboards

### 📌 Overview
To transition from SQL-based data transformation to visual storytelling, we export our aggregated DuckDB query results into **three lightweight, optimized CSV datasets**.

Rather than loading raw, unaggregated transactional logs into Tableau, pre-aggregating metrics in DuckDB ensures fast dashboard load times, accurate spatial calculations, and streamlined chart construction.

---

### 📁 Exported Summary Datasets & Visual Mapping

| Exported File | Core Dimension | Primary Metrics Included | Target Tableau Visualization |
| :--- | :--- | :--- | :--- |
| **`tableau_dim1_entrance_priming.csv`** | **Dimension 1:** Entrance Priming &amp; Sensory Architecture | `shopper_group`, `total_orders`, `avg_basket_size` | **Side-by-Side Bar Chart** comparing basket expansion (10.73 vs. 8.87 items). |
| **`tableau_dim2_3_departments.csv`** | **Dimensions 2 & 3:** Navigation Flow &amp; Category Anchors | `department`, `total_items_purchased`, `avg_item_pick_order` | **Lollipop / Timeline Chart** &amp; **2x2 Quadrant Matrix** (Volume vs. Pick Sequence). |
| **`tableau_dim4_spatial_zones.csv`** | **Dimension 4:** Front-End Queue &amp; Checkout Impulse | `store_zone`, `total_items_purchased`, `pct_total_volume`, `avg_item_pick_order` | **Donut Chart / Treemap** showing zone volume contributions (Entrance: 10.2%, Main: 82.8%, Checkout: 7.0%). |

---

### 💡 Methodological Takeaway
By establishing a clean pipeline between **Python (ingestion)** $\rightarrow$ **DuckDB (spatial SQL aggregations)** $\rightarrow$ **Tableau (interactive visual analytics)**, we create a scalable framework for physical retail layout optimization grounded in transaction sequence data.

```

In [5]:
import duckdb 

# Path to your processed sample file
csv_path = 'Data/spatial_transactions_sample.csv'  # Adjust path if your file is in the root folder

# --------------------------------------------------------------------------- #
# 1. Dimension 1: Entrance Priming Summary
# --------------------------------------------------------------------------- #
query_dim1 = f""" WITH order_summary AS ( SELECT order_id, COUNT(*) AS basket_size, MIN(CASE WHEN store_zone = 'Entrance_Sensory_Zone' THEN add_to_cart_order END) AS min_entrance_pick FROM '{csv_path}' GROUP BY order_id ) SELECT CASE WHEN min_entrance_pick IS NOT NULL AND min_entrance_pick <= 3 THEN 'Visited Entrance Zone First' ELSE 'Skipped Entrance Zone' END AS shopper_group, COUNT(order_id) AS total_orders, ROUND(AVG(basket_size), 2) AS avg_basket_size FROM order_summary GROUP BY shopper_group; """
df_dim1 = duckdb.sql(query_dim1).df()
df_dim1.to_csv('Data/tableau_dim1_entrance_priming.csv', index=False)

# --------------------------------------------------------------------------- #
# 2. Dimensions 2 & 3: Department Pick Order & Volume
# --------------------------------------------------------------------------- #
query_dim2_3 = f""" SELECT department, COUNT(*) AS total_items_purchased, ROUND(AVG(add_to_cart_order), 2) AS avg_item_pick_order FROM '{csv_path}' GROUP BY department ORDER BY avg_item_pick_order ASC; """
df_dim2_3 = duckdb.sql(query_dim2_3).df()
df_dim2_3.to_csv('Data/tableau_dim2_3_departments.csv', index=False)

# --------------------------------------------------------------------------- #
# 3. Dimension 4: Spatial Zone Volume Breakdown
# --------------------------------------------------------------------------- #
query_dim4 = f""" SELECT store_zone, COUNT(*) AS total_items_purchased, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM '{csv_path}'), 2) AS pct_total_volume, ROUND(AVG(add_to_cart_order), 2) AS avg_item_pick_order FROM '{csv_path}' GROUP BY store_zone ORDER BY avg_item_pick_order ASC; """
df_dim4 = duckdb.sql(query_dim4).df()
df_dim4.to_csv('Data/tableau_dim4_spatial_zones.csv', index=False)

print("✅ Success! All Tableau CSV summary files have been generated in your Data/ folder:")
print(" - Data/tableau_dim1_entrance_priming.csv")
print(" - Data/tableau_dim2_3_departments.csv")
print(" - Data/tableau_dim4_spatial_zones.csv")

✅ Success! All Tableau CSV summary files have been generated in your Data/ folder:
 - Data/tableau_dim1_entrance_priming.csv
 - Data/tableau_dim2_3_departments.csv
 - Data/tableau_dim4_spatial_zones.csv


In [6]:
import duckdb

csv_path = 'Data/spatial_transactions_sample.csv' 

# Build one master dataset with all order-level & transaction-level features
query_unified = f"""
 WITH order_features AS (
    SELECT 
        order_id, COUNT(*) AS basket_size, 
        CASE 
            WHEN MIN(CASE WHEN store_zone = 'Entrance_Sensory_Zone' THEN add_to_cart_order END) <= 3 
            THEN 'Visited Entrance Zone First'
            ELSE 'Skipped Entrance Zone'
        END AS shopper_group
    FROM '{csv_path}'
    GROUP BY order_id 
) SELECT 
    t.*, 
    f.basket_size,
    f.shopper_group 
FROM '{csv_path}' t 
JOIN order_features f ON t.order_id = f.order_id; 
""" 

df_unified = duckdb.sql(query_unified).df() 
df_unified.to_csv('Data/spatial_transactions_final.csv', index=False) 

print("✅ Success! Single master dataset saved to: Data/spatial_transactions_final.csv")

✅ Success! Single master dataset saved to: Data/spatial_transactions_final.csv


## 🚀 Strategic Recommendations & Call to Action

To translate these spatial insights into measurable business value, store operations and merchandising leadership should execute the following **4-Point Retail Layout Action Plan**: 

| # | Strategic Recommendation | Data-Backed Justification | Target Business Impact | 
| :-: | :--- | :--- | :--- | 
| **1** | **Protect & Expand Entrance Priming** | Entrance visitors generate a **+21.0% basket expansion** (+1.86 items/order). | Higher overall store revenue across all customer segments. | 
| **2** | **Align Merchandising with Pick Sequence** | Planned destination categories (*e.g., Alcohol*) are selected early (pick #5.38). | Smoother store flow and increased exposure to mid-trip categories. | 
| **3** | **Leverage Traffic Anchors for Cross-Selling** | *Produce* & *Dairy* drive **~46% of total store volume**. | Higher impulse conversion from captive foot traffic in core anchor zones. | 
| **4** | **Optimize Front-End Queue Merchandising** | Checkout queue displays capture **6.98% of item volume** at pick #14.39. | Maximized margin from captive waiting time in register queues. |